# Section 3 — FEMA hurricane filtering (Florida, county-year)

**Purpose**: extract Florida hurricane events from the FEMA disaster declarations dataset, aggregated to the county-year level.

**Input**: `DisasterDeclarationsSummaries.parquet` (all 69,942 records, 1953–today).

**Output**: `fema_fl_hurricanes_county_year.parquet` — one row per (Florida county, year) with hurricane events, indicating whether a hurricane hit that county that year.

**Sanity check target events**: Irma 2017 (Miami-Dade, Broward, Monroe); Ian 2022 (Lee, Charlotte). If these don't appear, something is wrong with the filter or the FIPS join.

This notebook is independent of Notebooks 01, 02, and 04.


## Setup

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("your/data/path/here")
FEMA_PARQUET = DATA_DIR / "DisasterDeclarationsSummaries.parquet"
OUT_FEMA_FL  = DATA_DIR / "fema_fl_hurricanes_county_year.parquet"

print(f"Input:  {FEMA_PARQUET}  (exists: {FEMA_PARQUET.exists()})")
print(f"Output: {OUT_FEMA_FL}")


## Load FEMA data

In [ ]:
fema = pd.read_parquet(FEMA_PARQUET)
print(f"Total FEMA records: {len(fema):,}")
print(f"Columns ({len(fema.columns)}): {list(fema.columns)}")

## Filter to hurricane / tropical storm events, Florida only

In [ ]:
# Filter to hurricane and tropical storm incident types
hurricanes = fema[fema['incidentType'].isin(['Hurricane', 'Tropical Storm'])].copy()
print(f"Hurricane + Tropical Storm records (all states): {len(hurricanes):,}")

# Filter to Florida
fl_hurricanes = hurricanes[hurricanes['state'] == 'FL'].copy()
print(f"Florida hurricane records: {len(fl_hurricanes):,}")

fl_hurricanes[['disasterNumber', 'declarationTitle', 'incidentType',
               'fipsStateCode', 'fipsCountyCode', 'incidentBeginDate']].head()


## Build 5-digit county FIPS and extract year

In [ ]:
# Concatenate 2-digit state FIPS + 3-digit county FIPS -> 5-digit STCOUNTYFP
# (matches the HUD crosswalk column of the same name)
fl_hurricanes['fipsStateCode']  = (
    fl_hurricanes['fipsStateCode'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.zfill(2)
)
fl_hurricanes['fipsCountyCode'] = (
    fl_hurricanes['fipsCountyCode'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.zfill(3)
)
fl_hurricanes['STCOUNTYFP'] = fl_hurricanes['fipsStateCode'] + fl_hurricanes['fipsCountyCode']

# Extract year from incident begin date
fl_hurricanes['incidentBeginDate'] = pd.to_datetime(fl_hurricanes['incidentBeginDate'])
fl_hurricanes['year'] = fl_hurricanes['incidentBeginDate'].dt.year

print("Sample after processing:")
fl_hurricanes[['disasterNumber', 'declarationTitle', 'STCOUNTYFP', 'year']].head()


## Aggregate to county-year

In [ ]:
county_year = (
    fl_hurricanes
    .groupby(['STCOUNTYFP', 'year'])
    .agg(
        n_hurricanes=('disasterNumber', 'nunique'),
        event_names=('declarationTitle', lambda x: ' | '.join(sorted(set(x))))
    )
    .reset_index()
)
county_year['hurricane_hit'] = 1

print(f"County-year rows: {len(county_year):,}")
print(f"Year range: {county_year['year'].min()} to {county_year['year'].max()}")
county_year.head()


## Sanity check: known events should appear

In [ ]:
# Irma 2017: expected in Miami-Dade (12086), Broward (12011), Monroe (12087)
irma_counties = ['12086', '12011', '12087']
irma_hits = county_year[
    (county_year['STCOUNTYFP'].isin(irma_counties)) & (county_year['year'] == 2017)
]
print(f"Irma 2017 (expected Miami-Dade, Broward, Monroe): {len(irma_hits)}/3 present")
print(irma_hits[['STCOUNTYFP', 'year', 'event_names']].to_string(index=False))
print()

# Ian 2022: expected in Lee (12071), Charlotte (12015)
ian_counties = ['12071', '12015']
ian_hits = county_year[
    (county_year['STCOUNTYFP'].isin(ian_counties)) & (county_year['year'] == 2022)
]
print(f"Ian 2022 (expected Lee, Charlotte): {len(ian_hits)}/2 present")
print(ian_hits[['STCOUNTYFP', 'year', 'event_names']].to_string(index=False))


In [ ]:
# Summary of hurricane events by year, 2015-2024
year_summary = (
    county_year[county_year['year'].between(2015, 2024)]
    .groupby('year')
    .agg(counties_hit=('STCOUNTYFP', 'nunique'),
         total_events=('n_hurricanes', 'sum'))
    .reset_index()
)
print("Florida hurricane events by year (2015-2024):")
print(year_summary.to_string(index=False))


## Save output

In [ ]:
county_year.to_parquet(OUT_FEMA_FL, index=False)
print(f"Saved: {OUT_FEMA_FL}")
print(f"Size:  {OUT_FEMA_FL.stat().st_size / 1e6:.2f} MB")
